# AI Job Search & Auto-Apply — Modular End-to-End Test Suite
This notebook verifies all core backend components of the **AI Job Search and Auto-Apply** application step-by-step **without requiring Streamlit**.

### Step-by-Step Architecture Pipeline Tested:
1. Environment & Path Initialization
2. Dynamic Candidate Profile & Config Loader
3. Dynamic System Prompt Generation (Bullet points, respectful tone, `\n\n` spacing)
4. Multi-format Database Loading (`.json`, `.csv`, `.xlsx`, `.txt`)
5. Strict Email Validation & Text Scraping (`is_valid_email`)
6. Recipient Salutation & Name Inference (`Hi [Name],`)
7. Email Body Quality Validator (`EmailValidator`)
8. Graceful Fallback Engine when LLM is Offline
9. Look-back Window & Deduplication Skip Logic
10. Multi-format Database Export (`.json`, `.csv`, `.xlsx`, `.txt`)
11. Domain-Wise Analytics & Domain Operations
12. End-to-End Streaming Batch Outreach Simulation (`run_email_batch_stream`)

## Step 1: Environment & Path Initialization
Ensure `src/` is in `sys.path` so all modular sub-packages can be imported cleanly.

In [ ]:
import sys
from pathlib import Path

# Find project root
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent if NOTEBOOK_DIR.name == "nb" else NOTEBOOK_DIR
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"[OK] Project Root: {PROJECT_ROOT}")
print(f"[OK] Source Directory: {SRC_DIR}")
assert SRC_DIR.is_dir(), "src/ directory must exist!"

## Step 2: Dynamic Candidate Profile & Config Loader
Verify `config_loader` loads settings from `data/config` dynamically.

In [ ]:
from config_loader import (
    get_candidate_profile,
    get_email_config,
    get_model_config,
    get_smtp_config,
)

profile = get_candidate_profile()
email_cfg = get_email_config()
model_cfg = get_model_config()
smtp_cfg = get_smtp_config()

print(f"[OK] Candidate Name: {profile.get('name')}")
print(f"[OK] Candidate Role: {profile.get('current_role')}")
print(f"[OK] Technical Skills Count: {len(profile.get('technical_skills', []))}")
print(f"[OK] Sender Email Config: {email_cfg.get('sender_email')}")
print(f"[OK] Model Name: {model_cfg.get('model_name')}")
print(f"[OK] SMTP Host: {smtp_cfg.get('host')}:{smtp_cfg.get('port')}")

## Step 3: Dynamic System Prompt Generation
Verify that the system prompt dynamically accesses `candidate_profile.json`, incorporates respectful tone guidelines, enforces clean bullet points (`•`), and mandates double newline (`\n\n`) section spacing.

In [ ]:
from ai_email_writer.prompts import build_generic_system_prompt, get_active_system_prompt

prompt = build_generic_system_prompt()
print("=== PREVIEW OF DYNAMIC GENERIC SYSTEM PROMPT ===")
print(prompt[:600] + "\n... [truncated] ...")

assert profile.get("name") in prompt, "Candidate name must appear in system prompt!"
assert "•" in prompt, "Bullet point instructions must be present!"
assert "\\n\\n" in prompt or "\n\n" in prompt, "Double newline separation must be enforced!"
print("\n[OK] Dynamic system prompt verified successfully!")

## Step 4: Multi-Format Database Parsing (.json, .csv, .xlsx, .txt)
Verify parsing databases across all supported formats with emails and last-sent timestamps.

In [ ]:
import io
import json
import pandas as pd
from email_db_helper import parse_email_database_content

# 1. Test JSON format
json_data = json.dumps({"alice@google.com": "2026-03-01", "bob@amazon.com": ""}).encode("utf-8")
parsed_json = parse_email_database_content(json_data, "test.json")
print(f"[OK] Parsed JSON: {parsed_json}")
assert "alice@google.com" in parsed_json

# 2. Test CSV format
csv_data = b"email,last_contacted_date\ncarol@meta.com,2026-02-15\ndave@apple.com,\n"
parsed_csv = parse_email_database_content(csv_data, "test.csv")
print(f"[OK] Parsed CSV: {parsed_csv}")
assert "carol@meta.com" in parsed_csv

# 3. Test TXT format
txt_data = b"eve@openai.com\t2026-01-10\nfrank@anthropic.com\n"
parsed_txt = parse_email_database_content(txt_data, "test.txt")
print(f"[OK] Parsed TXT: {parsed_txt}")
assert "eve@openai.com" in parsed_txt

## Step 5: Strict Email Validation & Text Scraping
Verify filtering out malformed email addresses and extracting valid ones from job postings.

In [ ]:
from email_db_helper import is_valid_email, extract_emails_from_text

# Validation tests
assert is_valid_email("recruiter@company.com") is True
assert is_valid_email("talent.acquisition@in.ey.com") is True
assert is_valid_email("invalid@domain.") is False
assert is_valid_email("not_an_email") is False
assert is_valid_email("spaces in@mail.com") is False

sample_job_post = """
We are hiring a Senior Data Scientist! Reach out to careers@ai-corp.com or
contact hiring.lead@techcorp.io. Please do not contact support@.invalid or test@.
"""

extracted = extract_emails_from_text(sample_job_post)
print(f"[OK] Extracted emails from job description: {extracted}")
assert "careers@ai-corp.com" in extracted
assert "hiring.lead@techcorp.io" in extracted
assert len(extracted) == 2

## Step 6: Recipient Salutation & Name Inference
Verify name extraction: personal emails produce `Hi [Name],` while generic mailboxes fall back to `Hi,`.

In [ ]:
from ai_email_writer.recipient import infer_recipient_name, get_greeting_name
from ai_email_writer.formatter import enforce_correct_greeting

name_vishal = infer_recipient_name("vishal.seth@capgemini.com")
first_name_vishal = get_greeting_name("vishal.seth@capgemini.com")
print(f"[OK] Inferred Name: {name_vishal} | First Name: {first_name_vishal}")
assert name_vishal == "Vishal Seth"
assert first_name_vishal == "Vishal"

# Greeting formatting
sample_msg = "I would like to apply for the Data Science role."
greeting_msg = enforce_correct_greeting(sample_msg, first_name_vishal)
print(f"[OK] Greeting Formatted Message:\n{greeting_msg}")
assert greeting_msg.startswith("Hi Vishal,")

name_hr = infer_recipient_name("hr@startup.com")
first_name_hr = get_greeting_name("hr@startup.com")
greeting_hr = enforce_correct_greeting(sample_msg, first_name_hr)
print(f"[OK] Generic Mailbox Greeting:\n{greeting_hr}")
assert name_hr is None
assert greeting_hr.startswith("Hi,")

## Step 7: Email Body Quality & Completeness Validator
Test the `EmailValidator` tool against complete vs defective email drafts.

In [ ]:
from ai_email_writer.validator import email_validator

candidate_name = profile.get("name", "Rahul Singh")

good_body = f"""Hi Vishal,

I am a Data Scientist with 3+ years of experience specializing in Machine Learning and Generative AI.
I would love to explore relevant data science roles at Capgemini.

My core technical strengths include:
• Python, PySpark, and SQL
• Generative AI, RAG, and LangChain
• Machine Learning and Predictive Analytics

I have attached my resume for your review. I would welcome a brief 10-minute introductory call at your convenience.

Best regards,
{candidate_name}"""

res_good = email_validator.validate(good_body, recipient_name="Vishal")
print(f"[OK] Complete Body Valid: {res_good.is_valid} | Issues: {res_good.issues}")
assert res_good.is_valid is True

bad_body = "Hey, hire me because [Company Name] is cool and I know things because"
res_bad = email_validator.validate(bad_body, recipient_name="Vishal")
print(f"[OK] Defective Body Valid: {res_bad.is_valid} | Issues detected: {res_bad.issues}")
assert res_bad.is_valid is False
assert len(res_bad.issues) > 0

## Step 8: Graceful Fallback Engine when LLM is Offline
Verify that when no LLM is detected or Ollama is offline, the system safely builds the formatted generic body.

In [ ]:
from ai_email_writer.generator import build_fallback_email

generic_body_sample = email_cfg.get("body_template", "I am reaching out regarding opportunities...")
fallback = build_fallback_email(
    recipient_email="priya.sharma@tcs.com",
    general_subject="Data Scientist Application",
    general_body=generic_body_sample,
)

print("=== FALLBACK EMAIL PREVIEW ===")
print(f"Subject: {fallback.subject}")
print(fallback.body[:300] + "...")
assert "Hi Priya" in fallback.body
print("[OK] Fallback engine produced correctly formatted email with recipient salutation!")

## Step 9: Look-Back Window & Deduplication Skip Logic
Verify that contacts reached within the look-back duration are flagged and skipped.

In [ ]:
from datetime import datetime, timedelta
from email_sender.date_utils import get_lookback_date

today = datetime.now().date()
one_month_ago = get_lookback_date(look_back=1, look_back_format="months")
three_days_ago = today - timedelta(days=3)
two_months_ago = today - timedelta(days=60)

# 3 days ago is inside 1 month lookback window -> should skip
assert three_days_ago >= one_month_ago
# 60 days ago is outside 1 month lookback window -> should not skip
assert two_months_ago < one_month_ago

print(f"[OK] Today: {today}")
print(f"[OK] Lookback Cutoff (1 month): {one_month_ago}")
print("[OK] Lookback date calculation verified successfully!")

## Step 10: Multi-Format Database Export (.json, .csv, .xlsx, .txt)
Test exporting the database into all four formats.

In [ ]:
from email_db_helper import export_email_database

sample_db = {
    "alex@capgemini.com": "2026-03-01",
    "beth@ey.com": "",
    "clark@google.com": "2026-02-10",
}

csv_out = export_email_database(sample_db, "csv")
xlsx_out = export_email_database(sample_db, "xlsx")
txt_out = export_email_database(sample_db, "txt")
json_out = export_email_database(sample_db, "json")

print(f"[OK] CSV bytes: {len(csv_out)} bytes")
print(f"[OK] Excel (XLSX) bytes: {len(xlsx_out)} bytes")
print(f"[OK] TXT bytes: {len(txt_out)} bytes")
print(f"[OK] JSON bytes: {len(json_out)} bytes")
assert b"alex@capgemini.com" in csv_out
assert b"alex@capgemini.com" in json_out

## Step 11: Domain-Wise Analytics & Domain Operations
Verify domain grouping, statistics, and bulk domain deletion.

In [ ]:
from email_db_helper import group_emails_by_domain, get_domain_statistics, delete_domain_from_database

test_network = {
    "rahul@capgemini.com": "2026-03-01",
    "amit@capgemini.com": "",
    "neha@in.ey.com": "2026-02-14",
    "vikram@fractal.ai": "",
}

stats = get_domain_statistics(test_network)
print(f"[OK] Total Emails: {stats['total_emails']}")
print(f"[OK] Total Domains: {stats['total_domains']}")
print(f"[OK] Contacted: {stats['contacted_count']} | Uncontacted: {stats['uncontacted_count']}")
assert stats["total_domains"] == 3

# Test domain deletion
updated, deleted = delete_domain_from_database(test_network, "capgemini.com")
print(f"[OK] Deleted {deleted} contacts from @capgemini.com. Remaining in DB: {len(updated)}")
assert deleted == 2
assert "rahul@capgemini.com" not in updated

## Step 12: End-to-End Streaming Batch Simulation
Simulate the streaming batch pipeline events (`init`, `email_start`, `llm_unavailable`, `email_complete`, `batch_complete`) without sending live SMTP emails.

In [ ]:
from email_sender.batch import run_email_batch_stream

recipients = ["test.candidate@domain.com", "recruiter.lead@tech.org"]

# We test the streaming generator initialization and step yield
stream = run_email_batch_stream(
    list_of_emails=recipients,
    subject="AI Engineer Application",
    body="Hi,\n\nI am a Data Scientist. Please review my attached resume.\n\nBest regards,\nRahul Singh",
    sender_email="candidate@gmail.com",
    sender_password="mock_password_for_test",
    attachments=None,
    look_back=1,
    look_back_format="months",
)

# Read the initial event
init_event = next(stream)
print(f"[OK] Received stream init event: {init_event}")
assert init_event["event"] == "init"
assert init_event["total_emails_in_db"] == 2
assert init_event["remaining_to_send"] == 2

print("\n============================================================")
print("ALL 12 TEST COMPONENTS PASSED END-TO-END SUCCESSFULLY!")
print("============================================================")